# R2-Dreamer Habitat Baseline — Evaluation with TopDownMap

**Goal:** Load a trained checkpoint, run 10 evaluation episodes, and visualize the agent trajectory + goal on a top-down map.

| Property | Value |
|----------|-------|
| Agent | R2-Dreamer (JAX, no goal conditioning) |
| Environment | HM3D ObjectNav (val split) |
| Visualization | TopDownMap with agent path + goal marker |
| Episodes | 10 |

In [ ]:
import os, sys, pickle
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

import jax
import jax.numpy as jnp

sys.path.insert(0, str(Path.cwd().parent.parent.parent))

import habitat
from habitat.utils.visualizations import maps

from modules.r2dreamer.agent import R2DreamerAgent
from modules.r2dreamer.config import R2DreamerConfig

plt.rcParams.update({"figure.dpi": 120})

In [ ]:
# --- Configuration ---
CHECKPOINT_DIR = Path("../../../output/r2dreamer-habitat-baseline/sanity-50k/checkpoints")
NUM_EPISODES = 10
MAX_STEPS = 500

# Find the latest checkpoint
ckpts = sorted(CHECKPOINT_DIR.glob("step_*.pkl"))
assert ckpts, f"No checkpoints found in {CHECKPOINT_DIR}"
ckpt_path = ckpts[-1]
print(f"Using checkpoint: {ckpt_path.name}")

In [ ]:
# --- Load agent from checkpoint ---
config = R2DreamerConfig(
    obs_shape=(3, 64, 64),
    num_actions=4,
)

rng_key = jax.random.PRNGKey(42)
rng_key, init_key = jax.random.split(rng_key)
agent = R2DreamerAgent(config, init_key)

with open(ckpt_path, "rb") as f:
    ckpt = pickle.load(f)

agent.params = jax.tree.map(jnp.array, ckpt["params"])
agent.slow_critic_params = jax.tree.map(jnp.array, ckpt["slow_critic_params"])
print(f"Loaded checkpoint from step {ckpt['step']}")

In [ ]:
# --- Create evaluation environment with TopDownMap ---
SCENE_DIR = Path("../../../data/scene_datasets/hm3d")
DATA_DIR = Path("../../../data/datasets/objectnav/hm3d/objectnav_hm3d_v2")

hab_cfg = habitat.get_config(
    "benchmark/nav/objectnav/objectnav_hm3d.yaml",
    overrides=[
        "+habitat/task/measurements@habitat.task.measurements.top_down_map=top_down_map",
    ],
)
with habitat.config.read_write(hab_cfg):
    hab_cfg.habitat.dataset.split = "val"
    hab_cfg.habitat.dataset.data_path = str(DATA_DIR / "val" / "val.json.gz")
    hab_cfg.habitat.dataset.scenes_dir = "data/scene_datasets"
    scene_cfg = next(SCENE_DIR.rglob("*scene_dataset_config.json"), None)
    if scene_cfg:
        hab_cfg.habitat.simulator.scene_dataset = str(scene_cfg)
    agent_cfg = hab_cfg.habitat.simulator.agents.main_agent
    agent_cfg.sim_sensors.rgb_sensor.height = 64
    agent_cfg.sim_sensors.rgb_sensor.width = 64
    hab_cfg.habitat.environment.max_episode_steps = MAX_STEPS

env = habitat.Env(config=hab_cfg)
print(f"Evaluation env ready — {len(env._dataset.episodes)} episodes available")

In [ ]:
ACTIONS = {0: "STOP", 1: "MOVE_FORWARD", 2: "TURN_LEFT", 3: "TURN_RIGHT"}

# --- Run evaluation episodes ---
results = []

for ep_idx in range(NUM_EPISODES):
    obs = env.reset()
    episode = env.current_episode
    goal_category = episode.goals[0].object_category if episode.goals else "unknown"

    frames_rgb = []
    frames_topdown = []
    actions_taken = []
    rewards = []

    # Reset agent RSSM state
    obs_dict = {
        "image": np.transpose(obs["rgb"][:, :, :3], (2, 0, 1)),
        "is_first": True,
    }

    for step in range(MAX_STEPS):
        rng_key, act_key = jax.random.split(rng_key)
        action = agent.act(obs_dict, act_key, training=False)  # greedy

        obs = env.step(action=action)
        metrics = env.get_metrics()

        rgb = obs["rgb"][:, :, :3]
        frames_rgb.append(rgb)
        actions_taken.append(action)

        # TopDownMap rendering
        if "top_down_map" in metrics:
            top_map = maps.colorize_draw_agent_and_fit_to_height(
                metrics["top_down_map"], rgb.shape[0]
            )
            frames_topdown.append(top_map)

        obs_dict = {
            "image": np.transpose(rgb, (2, 0, 1)),
            "is_first": False,
        }

        if env.episode_over:
            break

    success = metrics.get("success", 0.0)
    spl = metrics.get("spl", 0.0)
    dist = metrics.get("distance_to_goal", -1)

    results.append({
        "episode": ep_idx,
        "goal": goal_category,
        "steps": len(actions_taken),
        "success": success,
        "spl": spl,
        "final_dist": dist,
        "frames_rgb": frames_rgb,
        "frames_topdown": frames_topdown,
        "actions": actions_taken,
    })

    print(
        f"Episode {ep_idx}: goal={goal_category:>12s}  "
        f"steps={len(actions_taken):3d}  success={success:.0f}  "
        f"spl={spl:.3f}  dist={dist:.2f}m"
    )

## Summary

In [ ]:
successes = [r["success"] for r in results]
spls = [r["spl"] for r in results]
steps = [r["steps"] for r in results]

print(f"Episodes:     {len(results)}")
print(f"Success rate: {np.mean(successes)*100:.1f}%")
print(f"Mean SPL:     {np.mean(spls):.3f}")
print(f"Mean steps:   {np.mean(steps):.0f}")

## Episode Visualizations

For each episode: first-person RGB (left) and top-down map with agent trajectory (right) at key frames (start, 25%, 50%, 75%, end).

In [ ]:
for r in results:
    n = len(r["frames_rgb"])
    has_topdown = len(r["frames_topdown"]) > 0
    # Pick 5 keyframes evenly spaced
    indices = [0] + [int(n * f) for f in [0.25, 0.5, 0.75]] + [n - 1]
    indices = sorted(set(min(i, n - 1) for i in indices))

    ncols = len(indices)
    nrows = 2 if has_topdown else 1
    fig, axes = plt.subplots(nrows, ncols, figsize=(3 * ncols, 3 * nrows))
    if nrows == 1:
        axes = axes[np.newaxis, :] if ncols > 1 else np.array([[axes]])
    if ncols == 1:
        axes = axes[:, np.newaxis]

    status = "SUCCESS" if r["success"] else "FAIL"
    fig.suptitle(
        f"Episode {r['episode']} — goal: {r['goal']} — {status} "
        f"(steps={r['steps']}, spl={r['spl']:.3f})",
        fontsize=11, fontweight="bold",
    )

    for col, idx in enumerate(indices):
        axes[0, col].imshow(r["frames_rgb"][idx])
        action_name = ACTIONS.get(r["actions"][idx], "?")
        axes[0, col].set_title(f"t={idx} [{action_name}]", fontsize=8)
        axes[0, col].axis("off")

        if has_topdown and idx < len(r["frames_topdown"]):
            axes[1, col].imshow(r["frames_topdown"][idx])
            axes[1, col].axis("off")

    plt.tight_layout()
    plt.show()

## Action Distribution

In [ ]:
all_actions = [a for r in results for a in r["actions"]]
action_counts = {name: all_actions.count(idx) for idx, name in ACTIONS.items()}

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(action_counts.keys(), action_counts.values())
ax.set_ylabel("Count")
ax.set_title("Action Distribution Across All Episodes")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

In [ ]:
env.close()
print("Done.")